# 21 · ALM control − specificity

Is a PPC-silencing effect *specific* to PPC, or a generic consequence of
inactivating cortex / of the light itself? Two parts, **unilateral**
(`alm_control_uni`) and **bilateral** (`alm_control_bi`) ALM, both on Uniform.

Each part runs the **same four per-animal contrasts** (then folded WT vs HET,
exactly as the PPC notebook):
1. **within-phase** − ALM opto on vs off (laser randomised per trial − permutation + trial CI)
2. **between-phase, all trials** − ALM vs masking sessions (session-level dual CI)
3. **delta-of-deltas** − `[ALM(on−off)] − [masking(on−off)]`, ALM silencing beyond the light artifact
4. **ALM vs PPC-opto**, all trials − is the effect PPC-specific or generic cortical inhibition

Two extra summary stats ride through everything: **`reaction_time`** (recorded
median) and **`reaction_time_jitter`** (median after a per-trial U[0, 150) ms
draw, re-drawn each bootstrap resample − a recording-uncertainty check that
widens the RT interval without moving the RT difference).

In [ ]:
from shared_setup import *
from plotting.opto import plot_delta_swarm
apply_style()

experiment, info = load_data()
by_animal, groups = gather_genotypes(experiment)
het_ids, wt_ids = groups.get('het', []), groups.get('wt', [])
opto_ids = [a for a in OPTO_COHORT if a in experiment.animals]
print(f"opto cohort ({len(opto_ids)}): het={[a for a in opto_ids if by_animal.get(a)=='het']} "
      f"wt={[a for a in opto_ids if by_animal.get(a)=='wt']}")

In [ ]:
import math
DIST = 'Uniform'
STATS      = ['accuracy', 'hard_accuracy', 'easy_accuracy', 'recency', 'side_bias', 'psychometric']
STATS_RT   = STATS + ['reaction_time', 'reaction_time_jitter']   # RT stats ride through every contrast
SENSITIVITY = ['accuracy', 'hard_accuracy', 'easy_accuracy', 'sigma', 'recency', 'lapse_low', 'lapse_high']
BIAS        = ['mu', 'side_bias']
DISPLAY     = SENSITIVITY + BIAS
DISPLAY_RT  = DISPLAY + ['reaction_time', 'reaction_time_jitter']
N_PERM, N_BOOT = 100, 100
DU = ('trials', 'sessions')   # dual CI: trial (diagnostic) + session (honest) on between-phase / dod

def rows_from(diffs, aid):
    return collect_rows([{'stat': k, 'value': float(v)} for k, v in diffs.items()],
                        animal=aid, group=by_animal[aid])

def genotype_table(df):
    res = compare_groups(df, group_col='group')
    tbl = pd.DataFrame([{'stat': s, 'median_wt': r.get('median_a'), 'median_het': r.get('median_b'),
                         'p': r.get('p'), 'min_p': r.get('min_p')} for s, r in res.items()]
                       ).set_index('stat').reindex(DISPLAY_RT).round(4)
    return res, tbl

def swarm_grid(df, res, title):
    ncols = 4; nrows = math.ceil(len(DISPLAY_RT) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 3.3 * nrows), squeeze=False); axf = axes.ravel()
    for ax, stat in zip(axf, DISPLAY_RT):
        plot_delta_swarm(df, stat, ax=ax, p_value=res.get(stat, {}).get('p'),
                         group_col='group', value_col='value')
    for ax in axf[len(DISPLAY_RT):]: fig.delaxes(ax)
    fig.suptitle(title, fontsize=13); fig.tight_layout()

In [ ]:
# Helpers (flagged). The four ALM contrasts are identical for unilateral and
# bilateral, so the pipeline calls are written once here and invoked per `site`
# rather than duplicated across the two parts. Nothing hidden: the body is just
# the explicit compute_delta_stat / compute_interaction sequence.
def alm_contrasts(a, site, n_boot=N_BOOT, n_perm=N_PERM):
    alm = select_sessions(a, distribution=DIST, session_type=site)
    mk  = select_sessions(a, distribution=DIST, session_type='masking')
    op  = select_sessions(a, distribution=DIST, session_type='opto')
    a_on, a_off = filter_trials(alm, trial_type='opto'), filter_trials(alm, trial_type='non_opto')
    m_on, m_off = filter_trials(mk,  trial_type='opto'), filter_trials(mk,  trial_type='non_opto')
    a_all, m_all, o_all = (filter_trials(alm, trial_type='all'),
                           filter_trials(mk, trial_type='all'), filter_trials(op, trial_type='all'))
    r = {}
    if a_on and a_off:
        r['within'] = compute_delta_stat({'on': a_on, 'off': a_off}, stats=STATS_RT, reference='off',
                                         n_permutations=n_perm, n_bootstrap=n_boot, resample_units=DU)
    if m_on and m_off:
        r['within_masking'] = compute_delta_stat({'on': m_on, 'off': m_off}, stats=STATS_RT, reference='off',
                                         n_permutations=0, n_bootstrap=n_boot, resample_units=DU)
    if a_all and m_all:
        r['between'] = compute_delta_stat({'alm': a_all, 'masking': m_all}, stats=STATS_RT, reference='masking',
                                         n_permutations=0, n_bootstrap=n_boot, resample_units=DU)
    if n_boot > 0 and 'within' in r and 'within_masking' in r:
        r['dod'] = compute_interaction(r['within'], r['within_masking'], contrast='on_vs_off',
                                       label_a='alm', label_b='masking')
    if a_all and o_all:
        r['vs_ppc'] = compute_delta_stat({'alm': a_all, 'opto': o_all}, stats=STATS_RT, reference='opto',
                                         n_permutations=0, n_bootstrap=n_boot, resample_units=DU)
    return r

def alm_plot_animal(a, site, aid):
    r = alm_contrasts(a, site)
    specs = [('within',  'ALM opto on − off (within-phase effect)',    'cmp', ('trials',)),
             ('between', 'ALM vs masking (all trials, between-phase)',       'cmp', DU),
             ('dod',     'silencing beyond artifact (delta-of-deltas)',      'int', DU),
             ('vs_ppc',  'ALM vs PPC-opto (all trials)',                     'cmp', DU)]
    for key, title, kind, units in specs:
        if key not in r:
            print(f'{aid}: {key} unavailable (sessions missing)'); continue
        fig, axes = plt.subplots(3, 4, figsize=(13, 9)); axf = axes.ravel()
        for ax, stat in zip(axf, DISPLAY_RT):
            if kind == 'cmp': plot_stat_comparison_single(r[key], stat, ax=ax, units=units)
            else:             plot_interaction_single(r[key], stat, ax=ax, units=units)
        for ax in axf[len(DISPLAY_RT):]: fig.delaxes(ax)
        fig.suptitle(f"{aid} · {by_animal[aid]} · {DIST} · {title}", fontsize=12); fig.tight_layout()

def alm_fold(site):
    # Fold needs only per-animal point deltas, so no resampling (n_*=0). The
    # delta-of-deltas point is the difference of the two within-phase deltas.
    rows = {'within': [], 'between': [], 'dod': [], 'vs_ppc': []}
    n_seen = 0
    for aid in opto_ids:
        a = experiment.get_animal(aid)
        if not select_sessions(a, distribution=DIST, session_type=site):
            continue
        n_seen += 1
        r = alm_contrasts(a, site, n_boot=0, n_perm=0)
        if 'within' in r:  rows['within']  += rows_from(r['within']['contrasts']['on_vs_off']['diffs'], aid)
        if 'between' in r: rows['between'] += rows_from(r['between']['contrasts']['alm_vs_masking']['diffs'], aid)
        if 'vs_ppc' in r:  rows['vs_ppc']  += rows_from(r['vs_ppc']['contrasts']['alm_vs_opto']['diffs'], aid)
        if 'within' in r and 'within_masking' in r:
            wa = r['within']['contrasts']['on_vs_off']['diffs']
            wm = r['within_masking']['contrasts']['on_vs_off']['diffs']
            rows['dod'] += rows_from({s: wa[s] - wm.get(s, float('nan')) for s in wa}, aid)
    print(f'{site}: {n_seen} animals with ALM sessions')
    return rows

FOLD_TITLES = {'within': 'ALM opto on − off (within-phase)',
               'between': 'ALM vs masking (all trials)',
               'dod': 'silencing beyond artifact (delta-of-deltas)',
               'vs_ppc': 'ALM vs PPC-opto (all trials)'}

def alm_fold_report(site):
    fold = alm_fold(site)
    for k in ['within', 'between', 'dod', 'vs_ppc']:
        df = pd.DataFrame(fold[k])
        if not len(df):
            print(f'{site} {k}: no rows'); continue
        res, tbl = genotype_table(df)
        swarm_grid(df, res, f'{site} · {DIST} · {FOLD_TITLES[k]} — WT vs HET '
                            f'(n={df["animal"].nunique()})')
        display(tbl) if 'display' in dir() else print(tbl.to_string())

## Part A · Unilateral ALM (`alm_control_uni`)

Per-animal: the four contrasts. Then the WT-vs-HET fold, rank p + min_p, with the
n it actually ran on printed (only animals with `alm_control_uni` qualify).

In [ ]:
SITE_A = 'alm_control_uni'
ex_a = next((a for a in opto_ids
             if select_sessions(experiment.get_animal(a), distribution=DIST, session_type=SITE_A)), None)
if ex_a is None:
    print('No unilateral-ALM sessions in this cohort.')
else:
    alm_plot_animal(experiment.get_animal(ex_a), SITE_A, ex_a)

In [ ]:
alm_fold_report('alm_control_uni')

## Part B · Bilateral ALM (`alm_control_bi`)

Same four contrasts on bilateral ALM. Reaction time is the stat of interest here:
compare `reaction_time` (recorded) with `reaction_time_jitter` in each grid −
if the RT effect survives the jitter it is robust to the ±150 ms recording slop.

In [ ]:
SITE_B = 'alm_control_bi'
ex_b = next((a for a in opto_ids
             if select_sessions(experiment.get_animal(a), distribution=DIST, session_type=SITE_B)), None)
if ex_b is None:
    print('No bilateral-ALM sessions in this cohort.')
else:
    alm_plot_animal(experiment.get_animal(ex_b), SITE_B, ex_b)

In [ ]:
alm_fold_report('alm_control_bi')